# Nessie & Iceberg Demo - Versioning, Time Travel & Branching

This notebook demonstrates the key features of **Project Nessie** and **Apache Iceberg**:

1. **Git-like Versioning** - Every change is committed and tracked
2. **Time Travel** - Query data at any point in the past
3. **Branching** - Create isolated development environments
4. **Merging** - Promote changes from dev to production
5. **Rollback** - Revert to a previous state

**Prerequisites**: Notebooks 01-04 must have been executed to create the tables.

---
## INITIALIZATION

In [15]:
# Configure path to project
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import Spark modules and configuration
from lakehouse.spark_session import get_spark
from lakehouse.settings import AWS_BUCKET
from pyspark.sql.functions import current_timestamp, current_date, lit, col

# Create Spark session
spark = get_spark("nessie-demo")

# Configure Nessie main branch
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Create namespaces
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

print("=" * 60)
print("NESSIE & ICEBERG - VERSIONING DEMO")
print("=" * 60)

NESSIE & ICEBERG - VERSIONING DEMO


## PART 1: NESSIE CATALOG STRUCTURE

Let's start by exploring the current Nessie catalog structure.

In [16]:
# Display all namespaces in Nessie
print("=== Nessie Namespaces ===")
spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)

# List all tables
print("=== Tables in Bronze ===")
spark.sql("SHOW TABLES IN nessie.bronze").show(truncate=False)

print("=== Tables in Silver ===")
spark.sql("SHOW TABLES IN nessie.silver").show(truncate=False)

print("=== Tables in Gold ===")
spark.sql("SHOW TABLES IN nessie.gold").show(truncate=False)

=== Nessie Namespaces ===
+---------+
|namespace|
+---------+
|silver   |
|bronze   |
|gold     |
+---------+

=== Tables in Bronze ===
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|bronze   |sales    |false      |
+---------+---------+-----------+

=== Tables in Silver ===
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|silver   |sales    |false      |
+---------+---------+-----------+

=== Tables in Gold ===
+---------+------------------------+-----------+
|namespace|tableName               |isTemporary|
+---------+------------------------+-----------+
|gold     |sales_by_category_region|false      |
|gold     |sales_by_segment        |false      |
|gold     |top_products            |false      |
+---------+------------------------+-----------+



## PART 2: NESSIE BRANCHES

### Concept

**Nessie** brings Git-like versioning to data lake tables. Like Git for code, Nessie allows:

* creating isolated **branches** to experiment
* testing data transformations without impacting production
* **merging** validated changes into the main branch

```
┌─────────┐         ┌─────────┐
│  main   │ ←------ │   dev   │
│ (prod)  │  merge  │  (test) │
└─────────┘         └─────────┘
```

In [17]:
# Cleanup dev branch if it exists (for a clean demo)
spark.sql("DROP BRANCH IF EXISTS dev IN nessie")

print("=== 1. LIST NESSIE REFERENCES (BRANCHES) ===")
spark.sql("LIST REFERENCES IN nessie").show(truncate=False)

print("Explanation:")
print("- Displays all branches in the Nessie catalog")
print("- By default, only 'main' exists (production branch)")

=== 1. LIST NESSIE REFERENCES (BRANCHES) ===
+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |main|b0effe40888a1c55e1fa7ed001e43cfeed5363a022ce8773697085ae4edde328|
+-------+----+----------------------------------------------------------------+

Explanation:
- Displays all branches in the Nessie catalog
- By default, only 'main' exists (production branch)


In [18]:
# View active branch
spark.sql("USE REFERENCE main IN nessie")

print("=== 2. ACTIVE BRANCH ===")
spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

current_ref = spark.sql("SHOW REFERENCE IN nessie").first()[0]
print(f"We are on branch: {current_ref}")
print("All SQL operations will affect this branch.")

=== 2. ACTIVE BRANCH ===
+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |main|b0effe40888a1c55e1fa7ed001e43cfeed5363a022ce8773697085ae4edde328|
+-------+----+----------------------------------------------------------------+

We are on branch: Branch
All SQL operations will affect this branch.


### Creating a development branch

In [19]:
print("=== 3. CREATE BRANCH 'dev' ===")

try:
    spark.sql("CREATE BRANCH dev IN nessie FROM main").show(truncate=False)
    print("Branch 'dev' created successfully!")
except Exception as e:
    if "already exists" in str(e).lower():
        print("\nBranch 'dev' already exists (created previously)")
    else:
        print(f"\nInfo: {e}")

print("\nExplanation:")
print("- Branch 'dev' is a logical copy of 'main'")
print("- No physical data copy (instantaneous)")
print("- Changes on 'dev' will not impact 'main'")

=== 3. CREATE BRANCH 'dev' ===
+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |dev |b0effe40888a1c55e1fa7ed001e43cfeed5363a022ce8773697085ae4edde328|
+-------+----+----------------------------------------------------------------+

Branch 'dev' created successfully!

Explanation:
- Branch 'dev' is a logical copy of 'main'
- No physical data copy (instantaneous)
- Changes on 'dev' will not impact 'main'


### Switch to dev branch

In [20]:
print("\n=== 4. SWITCH TO BRANCH 'dev' ===")

spark.sql("USE REFERENCE dev IN nessie")

# Verify we are on dev
spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

print("Explanation:")
print("- We are now on branch 'dev'")
print("- All subsequent operations will affect 'dev'")
print("- Branch 'main' remains intact (production safe)")


=== 4. SWITCH TO BRANCH 'dev' ===


+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |dev |b0effe40888a1c55e1fa7ed001e43cfeed5363a022ce8773697085ae4edde328|
+-------+----+----------------------------------------------------------------+

Explanation:
- We are now on branch 'dev'
- All subsequent operations will affect 'dev'
- Branch 'main' remains intact (production safe)


### Modification on dev branch

Now that we are on `dev`, let's add data to test our pipeline. This change will not impact `main`.

In [21]:
print("=== 5. MODIFICATION ON BRANCH 'dev' ===")

# Check current state on dev
count_dev_before = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Bronze data on 'dev' before: {count_dev_before:,} records")

# Read and add a new batch on dev branch
batch_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_003.csv"

batch_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch_path)
)

batch_bronze = (
    batch_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_003.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_3"))
)

batch_bronze.writeTo("nessie.bronze.sales").append()

count_dev_after = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Bronze data on 'dev' after:  {count_dev_after:,} records")
print(f"Difference: +{count_dev_after - count_dev_before:,} records")

print("\nThis change is only on 'dev' - 'main' is not impacted!")

=== 5. MODIFICATION ON BRANCH 'dev' ===
Bronze data on 'dev' before: 6,865 records
Bronze data on 'dev' after:  10,365 records
Difference: +3,500 records

This change is only on 'dev' - 'main' is not impacted!


### Comparison main vs dev

Let's verify that `main` has not been modified by comparing data from both branches.

In [22]:
print("=== 6. COMPARISON: main vs dev ===")

# Data on dev (where we are)
dev_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]

# Switch to main to compare
spark.sql("USE REFERENCE main IN nessie")
main_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]

print(f"BRANCH 'dev' (with Batch 3): {dev_count:,} records")
print(f"BRANCH 'main' (production):  {main_count:,} records")
print(f"\nDifference: {dev_count - main_count:,} more records on 'dev'")

# Display batches present on main
print("Batches present on 'main':")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("Branch 'main' is intact! Changes on 'dev'")
print("have not impacted production.")

=== 6. COMPARISON: main vs dev ===
BRANCH 'dev' (with Batch 3): 10,365 records
BRANCH 'main' (production):  6,865 records

Difference: 3,500 more records on 'dev'
Batches present on 'main':
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
| batch_2| 3501|
+--------+-----+

Branch 'main' is intact! Changes on 'dev'
have not impacted production.


### Commit history

Let's display the commit history on the active branch, similar to `git log`.

In [23]:
print("=== 7. COMMIT HISTORY ON 'main' ===")
spark.sql("SHOW LOG IN nessie").show(truncate=False)

print("Explanation:")
print("- Each operation (CREATE, INSERT, MERGE) creates a commit")
print("- Commits have an ID, author, timestamp and message")
print("- Like Git, we can navigate the history!")

=== 7. COMMIT HISTORY ON 'main' ===
+------+---------+----------------------------------------------------------------+----------------------------------------------------+-----------+--------------------------+--------------------------+-----------------------------------------------------------------------------------------+
|author|committer|hash                                                            |message                                             |signedOffBy|authorTime                |committerTime             |properties                                                                               |
+------+---------+----------------------------------------------------------------+----------------------------------------------------+-----------+--------------------------+--------------------------+-----------------------------------------------------------------------------------------+
|enese |         |b0effe40888a1c55e1fa7ed001e43cfeed5363a022ce8773697085ae4edde328|Ic

## PART 3: TIME TRAVEL

With Iceberg, we can query data as it was at any point in the past.

In [24]:
# Display complete Bronze snapshot history
print("=== BRONZE SNAPSHOT HISTORY ===")
spark.sql("""
    SELECT 
        made_current_at,
        snapshot_id,
        parent_id
    FROM nessie.bronze.sales.history
    ORDER BY made_current_at
""").show(truncate=False)

=== BRONZE SNAPSHOT HISTORY ===
+-----------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |
+-----------------------+-------------------+-------------------+
|2026-03-20 11:50:07.891|5652987662729614333|NULL               |
|2026-03-20 11:50:10.479|944796840376479237 |5652987662729614333|
|2026-03-20 11:50:11.828|2241584426441775570|944796840376479237 |
|2026-03-20 12:15:00.89 |4955465065967609470|NULL               |
+-----------------------+-------------------+-------------------+



In [25]:
# Time Travel: Query data from a specific snapshot
first_snapshot_id = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.bronze.sales.history 
    ORDER BY made_current_at 
    LIMIT 1
""").first()[0]

print("\n=== TIME TRAVEL: DATA FROM FIRST SNAPSHOT ===")
count_at_first = spark.sql(f"""
    SELECT COUNT(*) as cnt 
    FROM nessie.bronze.sales VERSION AS OF '{first_snapshot_id}'
""").first()[0]

print(f"Number of records at first snapshot: {count_at_first:,}")

spark.sql(f"""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales VERSION AS OF '{first_snapshot_id}'
    GROUP BY batch_id
    ORDER BY batch_id
""").show()


=== TIME TRAVEL: DATA FROM FIRST SNAPSHOT ===
Number of records at first snapshot: 3,364
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
+--------+-----+



## PART 4: BRANCH MERGING

Once tests are validated on `dev`, we can merge the branch into `main`.

In [26]:
print("=== 8. MERGE 'dev' INTO 'main' ===")

# Merge dev into main
spark.sql("MERGE BRANCH dev INTO main IN nessie").show(truncate=False)

print("Merge successful! Changes from 'dev' are now in 'main'.")

# Verify main now has the data
spark.sql("USE REFERENCE main IN nessie")
main_count_after = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"'main' after merge: {main_count_after:,} records")

# Display batches present on main after merge
print("Batches present on 'main' after merge:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("Batch 3 is now present in production!")

=== 8. MERGE 'dev' INTO 'main' ===
+----+----------------------------------------------------------------+
|name|hash                                                            |
+----+----------------------------------------------------------------+
|main|2eb3401127c2c176db426a4a050b08d23bbba7760b47e61a7b4e732ea3154b1a|
+----+----------------------------------------------------------------+

Merge successful! Changes from 'dev' are now in 'main'.
'main' after merge: 10,365 records
Batches present on 'main' after merge:
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
| batch_2| 3501|
| batch_3| 3500|
+--------+-----+

Batch 3 is now present in production!


## PART 5: ROLLBACK - REVERT TO A PREVIOUS STATE

One of the most powerful advantages: we can revert to a previous state in case of error.

Now that we have merged the changes into `main`, let's demonstrate a rollback scenario.

In [27]:
# Current state before rollback
print("=== CURRENT STATE BEFORE ROLLBACK ===")
bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Bronze: {bronze_count:,} records")

# Get previous snapshot
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at
    FROM nessie.bronze.sales.snapshots
    ORDER BY committed_at DESC
""").collect()

if len(snapshots) >= 2:
    previous_snapshot_id = snapshots[1]['snapshot_id']
    
    print(f"\n=== ROLLBACK TO PREVIOUS SNAPSHOT ===")
    print(f"Target snapshot: {previous_snapshot_id}")
    
    # Read data at previous state
    bronze_previous = spark.sql(f"""
        SELECT * FROM nessie.bronze.sales VERSION AS OF '{previous_snapshot_id}'
    """)
    
    count_previous = bronze_previous.count()
    print(f"Data to restore: {count_previous:,} records")
    
    # Rollback: replace table content
    bronze_previous.writeTo("nessie.bronze.sales").using("iceberg").createOrReplace()
    
    print("\nBronze rollback completed!")
    
    # Verification
    count_after_rollback = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
    print(f"Bronze after rollback: {count_after_rollback:,} records")
    print(f"(Before rollback: {bronze_count:,})")

=== CURRENT STATE BEFORE ROLLBACK ===
Bronze: 10,365 records

=== ROLLBACK TO PREVIOUS SNAPSHOT ===
Target snapshot: 4955465065967609470
Data to restore: 6,865 records

Bronze rollback completed!
Bronze after rollback: 6,865 records
(Before rollback: 10,365)


## SUMMARY - KEY POINTS

In [28]:
print("""
╔══════════════════════════════════════════════════════════════╗
║          NESSIE & ICEBERG - DEMONSTRATED FEATURES           ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. TIME TRAVEL                                              ║
║     - Query data at a specific snapshot                      ║
║     - Compare snapshots to see changes                       ║
║                                                              ║
║  2. BRANCHING (Git-like for data)                            ║
║     - Create branches for isolated development               ║
║     - Make changes on dev without impacting main             ║
║     - Merge branches to promote to production                ║
║                                                              ║
║  3. VERSION CONTROL                                          ║
║     - Each write creates a new commit                        ║
║     - Complete history of all changes                        ║
║     - Return to any previous state                           ║
║                                                              ║
║  4. ACID TRANSACTIONS                                        ║
║     - Atomic commits (all or nothing)                        ║
║     - Consistent reads between snapshots                     ║
║     - Isolated branches for concurrent development           ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

# Return to main branch
spark.sql("USE REFERENCE main IN nessie")
print("Demo completed. Spark session connected to Nessie branch 'main'.")


╔══════════════════════════════════════════════════════════════╗
║          NESSIE & ICEBERG - DEMONSTRATED FEATURES           ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. TIME TRAVEL                                              ║
║     - Query data at a specific snapshot                      ║
║     - Compare snapshots to see changes                       ║
║                                                              ║
║  2. BRANCHING (Git-like for data)                            ║
║     - Create branches for isolated development               ║
║     - Make changes on dev without impacting main             ║
║     - Merge branches to promote to production                ║
║                                                              ║
║  3. VERSION CONTROL                                          ║
║     - Each write creates a new commit                        ║
║     - Complete history 